# MLD cross-validation

This notebook tunes the pruned maximum-likelihood decoder used by `decoder.py` on a deterministic subset of the labelled training data. Each candidate is evaluated separately for the `X` and `Z` bases using cross-validation, and the selected configuration minimizes the combined logical error rate (LER). Runtime is reported as a secondary metric.

The folds are used only for evaluation: MLD has no trainable parameters and is not fitted on the fold data.


In [1]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("installed dependencies for:", sys.executable)


installed dependencies for: <repo>/.venv/bin/python



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import time
from itertools import product

import numpy as np
import stim

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info
from mld import build_mld, marginals, mld_classp

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SUBSET_PER_BASIS = 5000
N_FOLDS = 3
SEED = 42
print("repository:", REPO_ROOT)
print("data:", DATA_PATH)


repository: <repo>
data: <repo>/data/train/train.csv


<repo>/submission_template/bp_osd.py:48: RuntimeWarning: overflow encountered in scalar multiply
  _CTZ_TABLE[((np.uint64(1) << np.uint64(_i)) * _DEBRUIJN) >> np.uint64(58)] = _i


## Load labelled data and MLD inputs

For each basis, the notebook loads detector events and logical labels, then extracts `H`, `L`, and `p` from the corresponding Stim circuit. A separate MLD table is built for each basis and each tested `hexp` value.


In [3]:
def load_basis_data(data_path: str | Path, subset_per_basis: int, seed: int):
    data = load_kaggle_csv(str(data_path))
    if "observable_flips" not in data:
        raise ValueError("cross-validation requires L0-L3 labels")
    result = {}
    for offset, basis in enumerate(("X", "Z")):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no {basis} rows found")
        rng = np.random.default_rng(seed + offset)
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": data["detector_events"][rows].astype(np.uint8),
            "labels": data["observable_flips"][rows].astype(bool),
        }
    return result

def load_problem_data(data_folder: str | Path):
    problems = {}
    for basis in ("X", "Z"):
        circuit = stim.Circuit.from_file(str(Path(data_folder) / f"{basis}_circuit.stim"))
        info = problem_info(circuit)
        problems[basis] = info
        print(f"{basis}: H={info['H'].shape}, L={info['L'].shape}, p={info['p'].shape}")
    return problems

basis_data = load_basis_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
problems = load_problem_data(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(basis, basis_data[basis]["events"].shape, basis_data[basis]["labels"].shape)


X: H=(70, 2470), L=(4, 2470), p=(2470,)
Z: H=(70, 2465), L=(4, 2465), p=(2465,)
X (5000, 70) (5000, 4)
Z (5000, 70) (5000, 4)


## Deterministic cross-validation folds

In [4]:
def make_folds(size: int, n_folds: int, seed: int) -> list[np.ndarray]:
    if n_folds < 2 or n_folds > size:
        raise ValueError("n_folds must be at least 2 and no larger than the subset size")
    indices = np.arange(size)
    np.random.default_rng(seed).shuffle(indices)
    return [fold.copy() for fold in np.array_split(indices, n_folds)]

folds = {
    basis: make_folds(len(basis_data[basis]["events"]), N_FOLDS, SEED + i)
    for i, basis in enumerate(("X", "Z"))
}
print({basis: [len(fold) for fold in folds[basis]] for basis in ("X", "Z")})


{'X': [1667, 1667, 1666], 'Z': [1667, 1667, 1666]}


## MLD parameter grid

Each tier is `(eps, max_states, threshold)`. The first tier always decodes every shot. A later tier re-decodes shots whose current logical marginal is between `threshold` and `1 - threshold`. The grid also tests the `hexp` heuristic and the lower-level `compact_every` performance parameter.


In [5]:
BASE_TIERS = ((1e-4, 1024, 0.0), (3e-5, 4096, 0.15))
BASE_HEXP = 0.5
BASE_COMPACT_EVERY = 32

# Tune the main two-pass decoder around the current defaults.
TWO_PASS_GRID = [
    {
        "tiers": ((first_eps, first_states, 0.0), (second_eps, second_states, threshold)),
        "hexp": hexp,
        "compact_every": compact_every,
    }
    for first_eps, first_states, second_eps, second_states, threshold, hexp, compact_every in product(
        (3e-4, 1e-4),
        (512, 1024),
        (1e-5, 3e-5),
        (4096, 8192),
        (0.15, 0.25, 0.35),
        (0.50, 0.60),
        (16, 32),
    )
]

# Test whether an additional expensive pass helps the hardest shots.
THREE_PASS_GRID = [
    {
        "tiers": (
            (3e-4, 512, 0.0),
            (3e-5, 4096, middle_threshold),
            (last_eps, last_states, final_threshold),
        ),
        "hexp": hexp,
        "compact_every": compact_every,
    }
    for middle_threshold, last_eps, last_states, final_threshold, hexp, compact_every in product(
        (0.15, 0.25),
        (1e-6, 3e-6),
        (4096, 8192),
        (0.25, 0.35),
        (0.50, 0.60),
        (16, 32),
    )
]

PARAM_GRID = []
_seen = set()
for candidate in TWO_PASS_GRID + THREE_PASS_GRID:
    key = repr(candidate)
    if key not in _seen:
        _seen.add(key)
        PARAM_GRID.append(candidate)

print(f"testing {len(PARAM_GRID)} MLD configurations")
print("baseline:", {"tiers": BASE_TIERS, "hexp": BASE_HEXP, "compact_every": BASE_COMPACT_EVERY})


testing 256 MLD configurations
baseline: {'tiers': ((0.0001, 1024, 0.0), (3e-05, 4096, 0.15)), 'hexp': 0.5, 'compact_every': 32}


In [6]:
TABLE_CACHE = {}

def get_table(basis: str, hexp: float):
    key = (basis, float(hexp))
    if key not in TABLE_CACHE:
        info = problems[basis]
        TABLE_CACHE[key] = build_mld(info["H"], info["L"], info["p"], hexp=hexp)
    return TABLE_CACHE[key]

def decode_mld(table: dict, detector_events: np.ndarray, tiers, compact_every: int) -> np.ndarray:
    events = np.asarray(detector_events, dtype=bool)
    marg = None
    for eps, max_states, threshold in tiers:
        if marg is None:
            classp, _ = mld_classp(table, events, eps=eps, max_states=max_states, compact_every=compact_every)
            marg = marginals(classp)
            continue
        selected = np.flatnonzero((np.abs(marg - 0.5) < 0.5 - threshold).any(axis=1))
        if len(selected):
            classp, _ = mld_classp(table, events[selected], eps=eps, max_states=max_states, compact_every=compact_every)
            marg[selected] = marginals(classp)
    return (marg > 0.5).astype(np.uint8)

def evaluate_configuration(params: dict) -> dict:
    basis_scores = {"X": [], "Z": []}
    total_wrong = 0
    total_bits = 0
    total_seconds = 0.0
    for basis in ("X", "Z"):
        events = basis_data[basis]["events"]
        labels = basis_data[basis]["labels"]
        table = get_table(basis, params["hexp"])
        for fold_index, test_indices in enumerate(folds[basis]):
            started = time.perf_counter()
            predictions = decode_mld(table, events[test_indices], params["tiers"], params["compact_every"])
            elapsed = time.perf_counter() - started
            score = logical_error_rate(predictions, labels[test_indices])
            wrong = int(np.count_nonzero(predictions.astype(bool) != labels[test_indices]))
            total_wrong += wrong
            total_bits += predictions.size
            total_seconds += elapsed
            basis_scores[basis].append({
                "fold": fold_index,
                "ler": score,
                "seconds": elapsed,
            })
    return {
        "params": params,
        "combined_ler": total_wrong / total_bits,
        "seconds": total_seconds,
        "basis": basis_scores,
    }


## Run the search

The objective is the combined LER across all held-out `X` and `Z` folds. Ties are broken by total runtime. The first run may spend extra time compiling the Numba kernel.


In [7]:
results = []
print(f"evaluating {len(PARAM_GRID)} MLD configurations")
for index, params in enumerate(PARAM_GRID, start=1):
    result = evaluate_configuration(params)
    results.append(result)
    print(
        f"{index}/{len(PARAM_GRID)} "
        f"LER={result['combined_ler']:.6f} "
        f"time={result['seconds']:.2f}s "
        f"params={params}"
    )

results.sort(key=lambda result: (result["combined_ler"], result["seconds"]))
best = results[0]
print("\nbest configuration:", best["params"])
print(f"best combined LER: {best['combined_ler']:.6f}")
print(f"best CV runtime: {best['seconds']:.2f}s")


evaluating 256 MLD configurations
1/256 LER=0.035225 time=11.60s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.15)), 'hexp': 0.5, 'compact_every': 16}
2/256 LER=0.035250 time=3.35s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.15)), 'hexp': 0.5, 'compact_every': 32}
3/256 LER=0.035250 time=2.25s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.15)), 'hexp': 0.6, 'compact_every': 16}
4/256 LER=0.035300 time=2.26s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.15)), 'hexp': 0.6, 'compact_every': 32}
5/256 LER=0.035425 time=3.32s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.25)), 'hexp': 0.5, 'compact_every': 16}
6/256 LER=0.035450 time=3.35s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.25)), 'hexp': 0.5, 'compact_every': 32}
7/256 LER=0.035400 time=2.49s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.25)), 'hexp': 0.6, 'compact_every': 16}
8/256 LER=0.035450 time=2.58s params={'tiers': ((0.0003, 512, 0.0), (1e-05, 4096, 0.25)), 'hexp':

In [8]:
for basis in ("X", "Z"):
    print(f"\n{basis} folds")
    for fold_result in best["basis"][basis]:
        print(fold_result)

print("\nUse these values in decoder.py:")
print("MLD_TIERS =", best["params"]["tiers"])
print("compact_every =", best["params"]["compact_every"])
print("hexp =", best["params"]["hexp"])
print("\nRerun the selected configuration on a fresh holdout before trusting the result.")



X folds
{'fold': 0, 'ler': 0.03284343131373725, 'seconds': 0.5897744590038201}
{'fold': 1, 'ler': 0.03674265146970606, 'seconds': 0.33202529099798994}
{'fold': 2, 'ler': 0.03226290516206483, 'seconds': 0.3672254589982913}

Z folds
{'fold': 0, 'ler': 0.03689262147570486, 'seconds': 0.6533914579995326}
{'fold': 1, 'ler': 0.03914217156568686, 'seconds': 0.3537161249987548}
{'fold': 2, 'ler': 0.032563025210084036, 'seconds': 0.4167277090018615}

Use these values in decoder.py:
MLD_TIERS = ((0.0001, 1024, 0.0), (3e-05, 4096, 0.15))
compact_every = 32
hexp = 0.5

Rerun the selected configuration on a fresh holdout before trusting the result.
